In [1]:
import os
import json
import sys
sys.path.append(os.path.abspath('../'))

from retrieval_models import *

with open('../data/data.json') as f:
    data = json.loads(f.read())

In [2]:
queries = data['queries_stopped_stemmed']
documents = data['documents_stopped_stemmed']
qrels = data['qrels']

print("Number of queries:", len(queries))
print("Number of documents,", len(documents))

Number of queries: 10
Number of documents, 229


In [3]:
def mean_per_key(scores: dict[str, dict[str, float]]) -> dict[str, float]:
    q_ids = [key for key in scores.keys()]
    key_lists = {k: [v] for k, v in scores[q_ids[0]].items()}
    for q_id in q_ids[1:]:
        q_scores = scores[q_id]
        for k, v in q_scores.items():
            key_lists[k].append(v)
    return {k: sum(v) / len(v) for k, v in key_lists.items()}

In [4]:
tf_results = compute_tf(queries, documents)
tf_scores = tf_results.compute_metrics(qrels)
mean_per_key(tf_scores)

{'map': 0.41563012421546974,
 'P_10': 0.41999999999999993,
 'recall_10': 0.71,
 'ndcg_cut_10': 0.43664253565172517}

In [5]:
bm25_results = compute_bm25(queries, documents)
bm25_scores = bm25_results.compute_metrics(qrels)
mean_per_key(bm25_scores)

{'map': 0.4453892473387232,
 'P_10': 0.43,
 'recall_10': 0.7333333333333334,
 'ndcg_cut_10': 0.47487553434605756}

In [6]:
ql_results = compute_ql(queries, documents)
ql_scores = ql_results.compute_metrics(qrels)
mean_per_key(ql_scores)

{'map': 0.4223295584957862,
 'P_10': 0.41,
 'recall_10': 0.7,
 'ndcg_cut_10': 0.4644782428298922}

In [7]:
def examine_results(results: RetrievalModelScores, q_id: str, top_k: int):
    query_text = queries[q_id]
    print("=" * 20)
    print(q_id)
    print(" ".join(query_text))
    print("=" * 20)
    
    doc_scores = results.doc_scores[q_id]
    score_pairs = sorted([(d_id, doc.score) for d_id, doc in doc_scores.items()], key=lambda x: x[1], reverse=True)[:top_k]
    for d_id, score in score_pairs:
        print(d_id, score)
        print(" ".join(documents[d_id]))
        scored_doc = doc_scores[d_id]
        print("Word scores:")
        for word, score in scored_doc.word_scores.items():
            print('\t', word, score)
        if scored_doc.missing_word_scores:
            print("Missing word scores:")
            for word, score in scored_doc.missing_word_scores.items():
                print(word, score, end=" ")
        print()
        print()

In [8]:
examine_results(tf_results, '1110199', 10)

1110199
wifi vs bluetooth
398442 8
Relat : wifi repeat wifi router wifi long rang antenna wifi rang extend wireless router bluetooth transmitt wifi extend outdoor wifi antenna wifi antenna .
Word scores:
	 Relat 0
	 : 0
	 wifi 7
	 repeat 0
	 router 0
	 long 0
	 rang 0
	 antenna 0
	 extend 0
	 wireless 0
	 bluetooth 1
	 transmitt 0
	 outdoor 0
	 . 0


554521 6
New Arrival . kind Tablet Pc Wifi Bluetooth Camera avail LightInTheBox . guarante cool Tablet Pc Wifi Bluetooth Camera high qualiti afford price . Check websit buy favorit Tablet Pc Wifi Bluetooth Camera . Also , kind awesom product avail LightInTheBox .
Word scores:
	 New 0
	 Arrival 0
	 . 0
	 kind 0
	 Tablet 0
	 Pc 0
	 Wifi 0
	 Bluetooth 0
	 Camera 0
	 avail 0
	 LightInTheBox 0
	 guarante 0
	 cool 0
	 high 0
	 qualiti 0
	 afford 0
	 price 0
	 Check 0
	 websit 0
	 buy 0
	 favorit 0
	 Also 0
	 , 0
	 awesom 0
	 product 0
	 bluetooth 3
	 wifi 3


8160527 5
Bluetooth 4.0 vs. Wi-Fi Direct : Speed Wi-Fi Direct promis device-to-devic tr

In [9]:
examine_results(bm25_results, '1110199', 10)

1110199
wifi vs bluetooth
398442 9.915324354459125
Relat : wifi repeat wifi router wifi long rang antenna wifi rang extend wireless router bluetooth transmitt wifi extend outdoor wifi antenna wifi antenna .
Word scores:
	 Relat 0
	 : 0
	 wifi 6.575486499785576
	 repeat 0
	 router 0
	 long 0
	 rang 0
	 antenna 0
	 extend 0
	 wireless 0
	 bluetooth 3.339837854673549
	 transmitt 0
	 outdoor 0
	 . 0


554521 9.720344089212906
New Arrival . kind Tablet Pc Wifi Bluetooth Camera avail LightInTheBox . guarante cool Tablet Pc Wifi Bluetooth Camera high qualiti afford price . Check websit buy favorit Tablet Pc Wifi Bluetooth Camera . Also , kind awesom product avail LightInTheBox .
Word scores:
	 New 0
	 Arrival 0
	 . 0
	 kind 0
	 Tablet 0
	 Pc 0
	 Wifi 0
	 Bluetooth 0
	 Camera 0
	 avail 0
	 LightInTheBox 0
	 guarante 0
	 cool 0
	 high 0
	 qualiti 0
	 afford 0
	 price 0
	 Check 0
	 websit 0
	 buy 0
	 favorit 0
	 Also 0
	 , 0
	 awesom 0
	 product 0
	 bluetooth 4.518292987905949
	 wifi 5.202051101

In [10]:
examine_results(ql_results, '1110199', 10)

1110199
wifi vs bluetooth
8160527 -21.25103561131846
Bluetooth 4.0 vs. Wi-Fi Direct : Speed Wi-Fi Direct promis device-to-devic transfer speed 250Mbps , Bluetooth 4.0 promis speed similar Bluetooth 3.0 25Mbps . Bluetooth 4.0 Wi-Fi Direct use 802.11 network standard reach maximum speed .
Word scores:
	 Bluetooth 0
	 4.0 0
	 vs. 0
	 Wi-Fi 0
	 Direct 0
	 : 0
	 Speed 0
	 promis 0
	 device-to-devic 0
	 transfer 0
	 speed 0
	 250Mbps 0
	 , 0
	 similar 0
	 3.0 0
	 25Mbps 0
	 . 0
	 use 0
	 802.11 0
	 network 0
	 standard 0
	 reach 0
	 maximum 0
	 bluetooth -2.357196562898399
	 vs -3.7460272600929394
	 wifi -7.573905894163561
Missing word scores:
wifi -7.573905894163561 

398442 -23.571876795909013
Relat : wifi repeat wifi router wifi long rang antenna wifi rang extend wireless router bluetooth transmitt wifi extend outdoor wifi antenna wifi antenna .
Word scores:
	 Relat 0
	 : 0
	 wifi -1.4106200746987119
	 repeat 0
	 router 0
	 long 0
	 rang 0
	 antenna 0
	 extend 0
	 wireless 0
	 bluetooth -

In [11]:
import numpy as np
k_1_range = [0, 0.6, 1.2, 1.8, 10]
b_range = np.linspace(0, 1.0, 5).tolist()
bm25_range_results = compute_bm25_range(queries, documents, k_1_range, b_range)
bm25_range_results

{0: {0.0: <retrieval_models.RetrievalModelScores at 0x76c251b82350>,
  0.25: <retrieval_models.RetrievalModelScores at 0x76c251a40280>,
  0.5: <retrieval_models.RetrievalModelScores at 0x76c251af20e0>,
  0.75: <retrieval_models.RetrievalModelScores at 0x76c2519abf40>,
  1.0: <retrieval_models.RetrievalModelScores at 0x76c251865de0>},
 0.6: {0.0: <retrieval_models.RetrievalModelScores at 0x76c25171fc40>,
  0.25: <retrieval_models.RetrievalModelScores at 0x76c2517d5ae0>,
  0.5: <retrieval_models.RetrievalModelScores at 0x76c25168b940>,
  0.75: <retrieval_models.RetrievalModelScores at 0x76c2515457e0>,
  1.0: <retrieval_models.RetrievalModelScores at 0x76c2515fb640>},
 1.2: {0.0: <retrieval_models.RetrievalModelScores at 0x76c2514b14e0>,
  0.25: <retrieval_models.RetrievalModelScores at 0x76c25136b220>,
  0.5: <retrieval_models.RetrievalModelScores at 0x76c251225180>,
  0.75: <retrieval_models.RetrievalModelScores at 0x76c2512e2fe0>,
  1.0: <retrieval_models.RetrievalModelScores at 0x76c2